# Boosting Step by Step: Predicting Customer Churn

Last month, a telecom operator lost more than a quarter of its customers. It would like to spot the customers likely to leave (*churn*) ahead of time, to make them a retention offer.

The plan:

1. Watch boosting correct its errors, tree after tree, on a single variable
2. Compare a decision tree, a random forest and boosting (CatBoost) on every variable
3. Understand the model's predictions, and play with them

No GPU needed: the whole notebook runs in under a minute.

Data: [Telco Customer Churn](https://github.com/IBM/telco-customer-churn-on-icp4d) (IBM, Apache 2.0 license), 7,043 fictional customers.

## Installation and Imports

In [ ]:
!pip install --quiet catboost

import time

import ipywidgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from matplotlib.ticker import PercentFormatter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

## The Data

In [ ]:
customers = pd.read_csv("https://raw.githubusercontent.com/shuuchuu/datasets/main/telco-churn/telco-churn.csv")

# Readable column names and values
columns = {
  "gender": "Gender",
  "SeniorCitizen": "Senior",
  "Partner": "Partner",
  "Dependents": "Dependents",
  "tenure": "Tenure (months)",
  "PhoneService": "Phone",
  "MultipleLines": "Multiple lines",
  "InternetService": "Internet",
  "OnlineSecurity": "Online security",
  "OnlineBackup": "Online backup",
  "DeviceProtection": "Device protection",
  "TechSupport": "Tech support",
  "StreamingTV": "TV streaming",
  "StreamingMovies": "Movie streaming",
  "Contract": "Contract",
  "PaperlessBilling": "Paperless billing",
  "PaymentMethod": "Payment method",
  "MonthlyCharges": "Monthly bill ($)",
  "TotalCharges": "Total billed ($)",
  "Churn": "Churned",
}
values = {
  "No phone service": "No phone",
  "No internet service": "No internet",
  "Fiber optic": "Fiber",
  "Month-to-month": "Month-to-month",
  "One year": "1 year",
  "Two year": "2 years",
  "Bank transfer (automatic)": "Automatic bank transfer",
  "Credit card (automatic)": "Automatic credit card",
}

customers = customers.drop(columns="customerID").rename(columns=columns).replace(values)
customers["Senior"] = customers["Senior"].map({0: "No", 1: "Yes"})
# New customers haven't paid anything yet: their total is blank
customers["Total billed ($)"] = pd.to_numeric(customers["Total billed ($)"], errors="coerce").fillna(0)
customers["Churned"] = (customers["Churned"] == "Yes").astype(int)

print(f"{len(customers)} customers, {customers['Churned'].mean():.1%} of whom churned")
customers.head()

A first look: the churn rate depending on a few variables.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5), sharex=True)
for ax, column in zip(axes, ["Contract", "Internet", "Payment method"]):
  rates = customers.groupby(column)["Churned"].mean().sort_values()
  ax.barh(rates.index, rates.to_numpy(), color="tab:red")
  ax.set_title(column)
  ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
axes[1].set_xlabel("Churn rate")
fig.tight_layout()

We set customers aside to evaluate the models honestly:

- **training** (60%): the models learn from these customers;
- **validation** (20%): we choose settings (the number of trees, for instance) on these customers;
- **test** (20%): we measure the final result on these customers, never seen until then.

In [ ]:
train, rest = train_test_split(customers, test_size=0.4, stratify=customers["Churned"], random_state=0)
validation, test = train_test_split(rest, test_size=0.5, stratify=rest["Churned"], random_state=0)
print(f"Training: {len(train)}, validation: {len(validation)}, test: {len(test)}")

## 1. Boosting Step by Step

To see what boosting does, we first keep a single variable: the monthly bill. The model has to predict the probability that a customer leaves from their bill alone.

We write the boosting ourselves, in a few lines:

1. start from a naive prediction: the average churn rate, for everyone;
2. a small tree (2 levels of questions) learns the **errors** of this prediction;
3. add a fraction of its correction to the prediction (the gradient descent "step");
4. a new tree learns the remaining errors, and so on.

In [ ]:
BILL = "Monthly bill ($)"
STEP = 0.3  # each tree only corrects part of the error
N_TREES = 200

x, y = train[[BILL]].to_numpy(), train["Churned"].to_numpy()
x_val, y_val = validation[[BILL]].to_numpy(), validation["Churned"].to_numpy()

trees = []
predictions = [np.full(len(y), y.mean())]  # naive prediction: the average rate
predictions_val = [np.full(len(y_val), y.mean())]
for _ in range(N_TREES):
  errors = y - predictions[-1]
  tree = DecisionTreeRegressor(max_depth=2).fit(x, errors)  # the tree learns the errors
  trees.append(tree)
  predictions.append(predictions[-1] + STEP * tree.predict(x))
  predictions_val.append(predictions_val[-1] + STEP * tree.predict(x_val))

# Mean (squared) error after 0, 1, 2… trees
train_error = [np.mean((y - p) ** 2) for p in predictions]
val_error = [np.mean((y_val - p) ** 2) for p in predictions_val]

Each dot is the churn rate observed in a $5 bill bracket. On the left, the model after a few trees; on the right, its errors in each bracket, and the next tree learning to correct them.

In [ ]:
grid = np.linspace(x.min(), x.max(), 500).reshape(-1, 1)
curves = np.cumsum([np.full(len(grid), y.mean())] + [STEP * t.predict(grid) for t in trees], axis=0)

brackets = pd.cut(train[BILL], bins=np.arange(15, 125, 5))
centers = np.array([b.mid for b in brackets.cat.categories])
in_bracket = brackets.cat.codes.to_numpy()
observed_rate = np.array([y[in_bracket == i].mean() if (in_bracket == i).any() else np.nan for i in range(len(centers))])


def plot_data(ax):
  ax.scatter(centers, observed_rate, color="tab:red", s=15, label="Observed churn rate")
  ax.set_xlabel(BILL)
  ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))


def show_step(n):
  fig, (left, right) = plt.subplots(1, 2, figsize=(14, 3.5))
  plot_data(left)
  left.plot(grid, curves[n], color="tab:blue", label=f"Model ({n} tree{'s' if n != 1 else ''})")
  left.set_title(f"Model after {n} tree{'s' if n != 1 else ''}")
  left.set_ylim(0, 0.6)
  left.legend(loc="upper left")
  bracket_errors = [(y - predictions[n])[in_bracket == i].mean() for i in range(len(centers))]
  right.bar(centers, bracket_errors, width=4, color="tab:gray", label="Mean error in the bracket")
  right.plot(grid, trees[n].predict(grid), color="tab:green", label=f"Tree #{n + 1}: learned correction")
  right.axhline(0, color="black", linewidth=0.5)
  right.set_title("Remaining errors, and the next tree learning them")
  right.set_xlabel(BILL)
  right.set_ylim(-0.3, 0.3)
  right.legend(loc="upper left")
  plt.show()


for n in range(3):
  show_step(n)

And with more trees? Move the slider: the curve fits the training data better and better… until it learns its noise. The error on the validation data goes back up: this is **overfitting**. The right number of trees is the one that minimizes the validation error.

In [ ]:
best = int(np.argmin(val_error))


def explore(n_trees):
  fig, (left, right) = plt.subplots(1, 2, figsize=(14, 4))
  plot_data(left)
  left.plot(grid, curves[n_trees], color="tab:blue", label="Model")
  left.set_title(f"Model after {n_trees} trees")
  left.set_ylim(0, 0.6)
  left.legend(loc="upper left")
  right.plot(train_error, label="Training")
  right.plot(val_error, label="Validation")
  right.axvline(n_trees, color="black", linestyle="--")
  right.scatter([best], [val_error[best]], color="tab:orange", zorder=3, label=f"Best: {best} trees")
  right.set_xlabel("Number of trees")
  right.set_ylabel("Mean error")
  right.legend()
  plt.show()


ipywidgets.interact(
  explore, n_trees=ipywidgets.IntSlider(value=1, min=0, max=N_TREES, description="Trees", continuous_update=False)
);

## 2. Tree, Forest or Boosting?

We now use every variable, and compare the three families of models seen in class on the same customers.

scikit-learn's trees only understand numbers: we turn each categorical variable ("Contract", "Internet"…) into 0/1 columns. CatBoost takes these variables as they are.

In [ ]:
features = [c for c in customers.columns if c != "Churned"]
categorical = customers[features].select_dtypes(exclude="number").columns.tolist()
as_numbers = pd.get_dummies(customers[features], columns=categorical, dtype=float)

y_train, y_val, y_test = train["Churned"], validation["Churned"], test["Churned"]
models, durations, probas = {}, {}, {}

start = time.perf_counter()
models["Decision tree"] = DecisionTreeClassifier(max_depth=5, random_state=0).fit(
  as_numbers.loc[train.index], y_train
)
durations["Decision tree"] = time.perf_counter() - start

start = time.perf_counter()
models["Random forest"] = RandomForestClassifier(
  n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=0
).fit(as_numbers.loc[train.index], y_train)
durations["Random forest"] = time.perf_counter() - start

for name in models:
  probas[name] = models[name].predict_proba(as_numbers.loc[test.index])[:, 1]

CatBoost chains up to 1,000 trees, keeping an eye on the error on the validation data. As in part 1, it ends up going back up: CatBoost keeps the model at the best number of trees.

In [ ]:
start = time.perf_counter()
catboost = CatBoostClassifier(iterations=1000, learning_rate=0.05, random_seed=0, verbose=False)
catboost.fit(
  train[features], y_train, cat_features=categorical, eval_set=(validation[features], y_val), use_best_model=True
)
durations["CatBoost"] = time.perf_counter() - start
models["CatBoost"] = catboost
probas["CatBoost"] = catboost.predict_proba(test[features])[:, 1]

history = catboost.get_evals_result()
plt.figure(figsize=(8, 3.5))
plt.plot(history["learn"]["Logloss"], label="Training")
plt.plot(history["validation"]["Logloss"], label="Validation")
plt.axvline(catboost.get_best_iteration(), color="tab:orange", linestyle="--", label="Best number of trees")
plt.xlabel("Number of trees")
plt.ylabel("Error")
plt.legend()
plt.show()

### Who Ranked the Test Customers Best?

We measure the **AUC**: pick at random a customer who left and one who stayed; the AUC is the probability that the model rates the one who left as more at risk. 50% is chance, 100% is a perfect model.

In [ ]:
pd.DataFrame(
  {
    "AUC (test)": {name: f"{roc_auc_score(y_test, p):.1%}" for name, p in probas.items()},
    "Training time": {name: f"{d:.1f} s" for name, d in durations.items()},
  }
)

The gaps are small here: with 7,000 customers and about twenty variables, the problem is simple, and even a single tree does well. On richer data (more customers, variables, interactions between them), boosting's lead is often clearer.

### And for the Business?

The retention team can't call everyone. If it calls customers in order of predicted risk, what share of the future churners does it reach?

In [ ]:
CALLED = 0.2

fig, ax = plt.subplots(figsize=(7, 5))
for name, p in probas.items():
  churned = y_test.to_numpy()[np.argsort(-p)]
  share_called = np.arange(1, len(churned) + 1) / len(churned)
  share_reached = np.cumsum(churned) / churned.sum()
  ax.plot(share_called, share_reached, label=name)
  reached = share_reached[int(CALLED * len(churned)) - 1]
  print(f"{name}: calling {CALLED:.0%} of the customers reaches {reached:.0%} of the churners")
ax.plot([0, 1], [0, 1], color="gray", linestyle="--", label="At random")
ax.axvline(CALLED, color="black", linewidth=0.5)
ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.set_xlabel("Share of customers called, from most to least at risk")
ax.set_ylabel("Share of churners reached")
ax.legend()
plt.show()

## 3. Understanding the Model, and Playing With It

Which variables does the model rely on most?

In [ ]:
importances = pd.Series(catboost.get_feature_importance(), index=features).sort_values()
importances.plot.barh(figsize=(7, 5), color="tab:blue", title="Variable importance (CatBoost)")
plt.show()

Why is this customer rated at risk? For each customer, CatBoost can break its prediction down into contributions from each variable ([Shapley values](https://en.wikipedia.org/wiki/Shapley_value)): in red, what pushes toward leaving; in green, what keeps the customer.

In [ ]:
# A customer rated at risk, without being the most extreme case
customer = test.index[np.argsort(probas["CatBoost"])[int(0.9 * len(test))]]

contributions = catboost.get_feature_importance(
  Pool(test.loc[[customer], features], cat_features=categorical), type="ShapValues"
)[0, :-1]
contributions = pd.Series(contributions, index=[f"{f} = {test.loc[customer, f]}" for f in features])
top = contributions[contributions.abs().sort_values().index[-8:]]

ax = top.plot.barh(figsize=(8, 4), color=["tab:red" if c > 0 else "tab:green" for c in top])
ax.set_xticks([])
plt.title(f"Predicted churn probability: {catboost.predict_proba(test.loc[[customer], features])[0, 1]:.0%}")
plt.xlabel("← keeps the customer            pushes toward leaving →")
plt.show()

What if their situation changed? Change their contract, services or bill, and watch their churn probability move.

In [ ]:
def simulate(contract, internet, tech_support, payment, tenure, bill):
  changed = test.loc[[customer], features].copy()
  changed["Contract"] = contract
  changed["Internet"] = internet
  changed["Tech support"] = tech_support if internet != "No" else "No internet"
  changed["Payment method"] = payment
  changed["Tenure (months)"] = tenure
  changed["Monthly bill ($)"] = bill
  changed["Total billed ($)"] = tenure * bill
  proba = catboost.predict_proba(changed)[0, 1]
  fig, ax = plt.subplots(figsize=(8, 0.8))
  ax.barh([0], [proba], color="tab:red" if proba > 0.5 else "tab:green")
  ax.set_xlim(0, 1)
  ax.set_yticks([])
  ax.xaxis.set_major_formatter(PercentFormatter(1, decimals=0))
  ax.axvline(initial, color="black", linestyle="--")
  ax.set_title(f"Churn probability: {proba:.0%} (initially: {initial:.0%})")
  plt.show()


current = test.loc[customer]
initial = probas["CatBoost"][test.index.get_loc(customer)]
ipywidgets.interact(
  simulate,
  contract=ipywidgets.Dropdown(options=["Month-to-month", "1 year", "2 years"], value=current["Contract"], description="Contract"),
  internet=ipywidgets.Dropdown(options=["Fiber", "DSL", "No"], value=current["Internet"], description="Internet"),
  tech_support=ipywidgets.Dropdown(
    options=["Yes", "No"], value=current["Tech support"] if current["Internet"] != "No" else "No", description="Support"
  ),
  payment=ipywidgets.Dropdown(
    options=sorted(customers["Payment method"].unique()), value=current["Payment method"], description="Payment"
  ),
  tenure=ipywidgets.IntSlider(value=current["Tenure (months)"], min=0, max=72, description="Tenure", continuous_update=False),
  bill=ipywidgets.FloatSlider(
    value=current["Monthly bill ($)"], min=18, max=120, step=1, description="Bill ($)", continuous_update=False
  ),
);

## Key Takeaways

- Boosting chains small trees, each learning to correct the errors of the previous ones.
- Too many trees lead to overfitting: the number of trees is chosen on validation data, never on test data.
- On tabular data (customer files, transactions…), boosting is very often the best choice; CatBoost handles categorical variables directly.
- A technical score (the AUC) translates into a business decision: how many churners we reach by calling how many customers.
- Beware: the model finds correlations, not causes. Having a customer sign a 2-year contract won't necessarily make them loyal. And should age ("Senior") or gender influence who gets an offer?